# GPW multi-dataset experiments: Kaggle

Enable Internet and a GPU in Kaggle notebook settings. Attach the private
`gpw-kaggle-inputs` dataset containing airline.npz, airline.json and
icethk.txt from the supplied input bundle. Set INPUT_ROOT to its exact
mounted directory if it differs. All three files are checksum-verified;
no file search, unpickling or substitution occurs. The NPZ is the explicitly
approved one-time conversion of your trusted pickle; its original is unchanged.
Other real datasets download from the approved, pinned sources.

Code loads latest main of chahineNejm/Generalized-Patchwork. Every stage
uses a fresh worker, so setup can be rerun without restarting the kernel.
Model/data code is never embedded in this notebook.

The reduced batch selects 11 dataset configurations and 61 model runs.
FLIGHT and VNNGP are excluded. Each synthetic function uses D=5 and
50000 training points. All existing GPW/ProSpar sizes are retained;
Standard Patchwork uses only its PC-tree variant. Full presets remain
available for explicit later selection. Training is OFF until
RUN_TRAINING=True. Review the complete plan first.
No point budget, dataset size, device or numerical value is reduced silently.

New real and synthetic protocols use seeded 80/20; existing OISST/UKHousing
protocols and official MODIS split/training-only detrending are retained.
All models keep their approved ARD RBF/initialization/optimizer settings.
Standard Patchwork uses all input dimensions and paper-style balanced
PC trees. Grid/Voronoi variants are deferred from the current selection. Reduced-data
Standard Patchwork uses exactly the primary GPW initial selected targets,
while GPW/SVGP/ProSpar/VNNGP train on all observations.

ProSpar is deliberately smaller than its paper: matched budgets here are
512/1024, versus the paper's 200x25=5000 points. Actual/paper capacities
are recorded with every ProSpar result. These are controlled comparisons,
not reproductions of every paper's kernel, split or optimization protocol.

Precipitation and satellite drag are explicitly DEFERRED: verified source
data are unavailable. They are not silently skipped from a selected plan.
Additional synthetic dimensions/sample sizes are deferred, not deleted.
The selected 80/20 synthetic test size is12500, unlike the paper's25000.

Train for at most100 epochs with20 warm-up. After warm-up, stop after10
epochs without a relative training-loss improvement of at least0.1%.
Epoch21 establishes the monitoring baseline. Patience ignores warm-up;
no validation/test targets are used. Evaluate the last trained parameters,
never restore an earlier checkpoint. Actual epoch counts and stopping
reasons are saved. EPOCHS/WARMUP_EPOCHS/EARLY_STOPPING remain editable;
set EARLY_STOPPING=None to explicitly disable stopping for a smoke run.

Results, plan, worker logs, tracebacks and per-run ZIPs are written under
/kaggle/working/gpw_results. Every run saves source/data provenance, full
effective configuration, initial/final checkpoints, predictions, metrics,
timing, training/kernel histories and optional full-dimensional positions.
Metrics include both explicitly labeled standardized and original-target
scales; do not compare values with different units as the same metric.
High-dimensional location plots are labeled two-feature projections.
Each completed model is saved and archived before the next starts, with
a download link displayed immediately. Completed archives survive a later
model failure within the session; download them before session storage is
lost. This does not create an external backup automatically.
Save a Kaggle version with outputs to retain them beyond the session.


## 1. Latest source and dependencies


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/kaggle/working/gpw_source')
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if ROOT.exists():
    top = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--show-toplevel"], text=True).strip()
    if Path(top).resolve() != ROOT.resolve():
        raise ValueError("Existing checkout must be its own Git repository")
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    if remote != REPOSITORY or branch != BRANCH:
        raise ValueError("Existing checkout repository or branch does not match the requested source")
    subprocess.run(["git", "-C", str(ROOT), "diff", "--exit-code", "HEAD"], check=True)
    untracked = subprocess.check_output(["git", "-C", str(ROOT), "ls-files", "--others",
        "--exclude-standard", "--", ":(glob)src/**/*.py", ":(glob)experiments/**/*.py"], text=True).strip()
    if untracked:
        raise ValueError(f"Untracked source files prevent verified reuse: {untracked}")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge-base", "--is-ancestor", "HEAD", "origin/" + BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
                    "--single-branch", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)

COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
for path in reversed((str(ROOT / "src"), str(ROOT))):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[data,ukhousing,benchmark]"], check=True)
import runpy
run_worker = runpy.run_path(str(ROOT / "experiments/colab_client.py"))["run_worker"]
WORK_ROOT = Path("/kaggle/working/gpw_results")
DATA_ROOT = Path("/kaggle/working/gpw_data")

def call_worker(action, *, on_event=None, **values):
    return run_worker(ROOT, {"action": action, "source_provenance": SOURCE_PROVENANCE, **values},
        module="experiments.multidataset_worker", jobs_directory=WORK_ROOT / "worker_jobs", on_event=on_event)
_OISST_STAGES['install'] = True


## 2. Dataset/model selection and all call-time overrides

Edit DATASETS to choose/order datasets. MODELS_BY_DATASET selects/orders
model names shown in AVAILABLE_MODELS; an absent entry means the complete
displayed preset minus EXCLUDED_MODEL_TYPES. VNNGP is excluded by default;
an explicit selection conflicting with an exclusion raises.
Unknown names raise.
MODEL_OVERRIDES is {dataset: {model_name: {field: value}}}.
DATA_OVERRIDES is {dataset: {field: value}}; no source/schema relabeling.

Example short selection: DATASETS=["california"] and
MODELS_BY_DATASET={"california": ["SVGP", "GPW"]}.
For a long run, explicitly change EPOCHS and WARMUP_EPOCHS; no automatic
warm-up shrinking occurs. Rerunning training always starts fresh.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
from datetime import UTC, datetime
from IPython.display import FileLink, Image, Markdown, display
import pandas as pd

INPUT_ROOT = "/kaggle/input/datasets/chahinenejma/gpw-kaggle-inputs"
DATASETS = ['oisst', 'ukhousing', 'modis', 'california', 'tco', 'tco_l2', 'icethick', 'protein', 'sarcos', 'michalewicz_d5_n50000', 'g_function_d5_n50000']
EXCLUDED_MODEL_TYPES = ["vnngp"]
MODELS_BY_DATASET = {
    "oisst": [
        "SVGP",
        "GPW",
        "ProSpar",
        "Standard Patchwork (PC tree)"
    ],
    "ukhousing": [
        "SVGP",
        "GPW",
        "GPW 4 x 256",
        "ProSpar 8 x 128",
        "ProSpar 4 x 256",
        "Standard Patchwork (PC tree)"
    ],
    "modis": [
        "SVGP",
        "GPW",
        "GPW 4 x 120"
    ],
    "california": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "tco": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "tco_l2": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "icethick": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "protein": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "sarcos": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "michalewicz_d5_n50000": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "g_function_d5_n50000": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ]
}
MODEL_OVERRIDES = {}
DATA_OVERRIDES = {}
EPOCHS = 100
WARMUP_EPOCHS = 20
EARLY_STOPPING = {'patience': 10, 'relative_min_improvement': 0.001}
SAVE_INDUCING_POSITIONS = True

settings = call_worker("settings", device="cuda:0", input_root=INPUT_ROOT, datasets=DATASETS,
    excluded_model_types=EXCLUDED_MODEL_TYPES,
    early_stopping=EARLY_STOPPING,
    models=MODELS_BY_DATASET, model_overrides=MODEL_OVERRIDES, data_overrides=DATA_OVERRIDES,
    epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS, save_inducing_positions=SAVE_INDUCING_POSITIONS)
PLAN, PLAN_SHA256 = settings["plan"], settings["plan_sha256"]
AVAILABLE_MODELS = settings["available"]
display(pd.DataFrame(settings["summary"]))
print("Total selected model runs:", settings["total_runs"])
print("Total training epochs across runs:", settings["total_epochs"])
print("Deferred datasets:", PLAN["deferred"])
display(pd.DataFrame([{"dataset": row["dataset"], "model": row["name"], **row["config"]} for row in PLAN["runs"]]))
_GPW_CELL_RESULT = print("Available model names:", AVAILABLE_MODELS)
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


## 3. Check attached inputs, requested GPU and available disk


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREFLIGHT_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("preflight_%Y%m%dT%H%M%S%fZ")
preflight = call_worker("preflight", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREFLIGHT_DIRECTORY))
_GPW_CELL_RESULT = print(preflight)
_OISST_STAGES['preflight'] = True
_GPW_CELL_RESULT


## 4. Prepare selected datasets once or explicitly load prepared files

Leave this stage untouched when only model settings change. Use
PREPARED_ACTION="load" and an exact PREPARED_PATH to reuse saved arrays;
configuration/checksum mismatches raise without rebuilding or filling.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREPARED_ACTION = "prepare"
PREPARED_PATH = None
if PREPARED_ACTION == "prepare":
    PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("prepared_%Y%m%dT%H%M%S%fZ")
    prepared = call_worker("prepare", plan=PLAN, plan_sha256=PLAN_SHA256,
        directory=str(PREPARED_DIRECTORY), data_root=str(DATA_ROOT))
elif PREPARED_ACTION == "load":
    if PREPARED_PATH is None:
        raise ValueError("Set PREPARED_PATH explicitly for verified prepared-data reuse")
    PREPARED_DIRECTORY = Path(PREPARED_PATH)
    prepared = call_worker("load", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREPARED_DIRECTORY))
else:
    raise ValueError("PREPARED_ACTION must be prepare or load")
counts = []
for spec in PLAN["data"]:
    import numpy as np
    with np.load(PREPARED_DIRECTORY / spec["name"] / "split.npz", allow_pickle=False) as rows:
        counts.append({"dataset": spec["name"], **{label: len(rows[label]) for label in ("train", "validation", "test")}})
display(pd.DataFrame(counts))
_GPW_CELL_RESULT = print("Prepared datasets:", prepared["directory"])
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 5. Explicitly authorize and run the displayed plan

RUN_TRAINING=False intentionally raises before any model starts. Set it
True only after reviewing selection, counts and capacities above. There
is no endless sweep: exactly the selected configurations and epoch counts
run. Any failure stops subsequent work with a full traceback. Interrupting
the notebook terminates the worker. Completed results/ZIPs remain on disk.
Changing the selection/hyperparameters requires rerunning settings and
preflight; unchanged prepared data can be reused explicitly.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
RUN_TRAINING = False
if RUN_TRAINING is not True:
    raise RuntimeError("Training disabled. Review the plan, then explicitly set RUN_TRAINING=True")
live = display("Starting selected experiments...", display_id=True)

def show_event(event):
    path = Path(event["directory"])
    if event["kind"] == "snapshot":
        live.update(Image(filename=str(path / "visuals/training.png")))
    elif event["kind"] == "complete":
        display(Markdown("### " + event["dataset"] + " / " + event["name"]))
        display(pd.DataFrame([event["metrics"]]).T)
        display(Image(filename=str(path / "visuals/predictions.png")))
        display(Image(filename=str(path / "visuals/hyperparameters.png")))
        metadata = json.loads((path / "model_metadata.json").read_text())
        if metadata["inducing_positions_saved"]:
            display(Image(filename=str(path / "visuals/inducing_positions.png")))
        print("Saved completed-run archive:", event["archive"])
        display(FileLink(event["archive"]))
    else:
        raise ValueError(f"Unknown worker event: {event['kind']}")

BATCH_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("experiments_%Y%m%dT%H%M%S%fZ")
result = call_worker("train", plan=PLAN, plan_sha256=PLAN_SHA256,
    prepared_directory=str(PREPARED_DIRECTORY), directory=str(BATCH_DIRECTORY),
    execute_training=RUN_TRAINING, on_event=show_event)
display(pd.read_csv(Path(result["directory"]) / "comparison.csv"))
_GPW_CELL_RESULT = print("Complete collection:", result["directory"])
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 6. Archive the completed collection


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['archive']:
    _OISST_STAGES[_stage] = False
for _required in ['training']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
ARCHIVE_PATH = WORK_ROOT / datetime.now(UTC).strftime("all_results_%Y%m%dT%H%M%S%fZ.zip")
archived = call_worker("archive", plan=PLAN, plan_sha256=PLAN_SHA256,
    directory=str(BATCH_DIRECTORY), destination=str(ARCHIVE_PATH))
from IPython.display import FileLink
display(FileLink(archived["archive"]))
print("Worker requests/logs/failures:", WORK_ROOT / "worker_jobs")
_GPW_CELL_RESULT = print("Save a Kaggle version with outputs to preserve all files.")
_OISST_STAGES['archive'] = True
_GPW_CELL_RESULT
